# Reverse-Engineering the EMMa Dataset Search Files

This notebook documents the process of reverse-engineering and reconstructing
the metadata needed to work with the **EMMa** dataset:

> Standley et al., *"An Extensible Multimodal Multi-task Object Dataset with Materials"*

The dataset provides:

- A main `.fdd` file with **per-object attributes** (materials distribution,
  category IDs, price, mass, ratings, custom properties, EM flags, etc.).
- A `search_file.fdd` containing a big search string for each ASIN (title,
  description, features, materials, categories) separated by `|||`.
- Several auxiliary search / index files (`*_search_file.pkl`) that allow fast
  lookup by category, material bitfield, EM / listed bits, etc.

However, the dataset **does not** provide mappings for:

- Category IDs → category names
- Material indices → material names
- Custom property indices → property names

It also doesn’t document the exact meaning / encoding of the `listed_search`,
`material_search`, and `custom_search` bitfields.

In this notebook we:

1. Explore the contents of the main dataset and search files to understand
   their structure and how they relate.
2. Build the **category ID → name** mapping (`cat_id2name.pkl`) using the
   dataset’s category IDs plus the text fields in `search_file.fdd`.
3. Build the **material index → material name** list (`material_names.pkl`) by
   actually *deriving* the mapping using the 182-dimensional materials
   probability vector, the 23-byte bitfield in `material_search`, and the
   materials segment of `search_file.fdd`.
4. Build the **custom index → custom name** list (`custom_names.pkl`) by
   correlating the 5D `custom` probabilities with the 8-bit `custom_search`
   field.
5. Build an optional **global sort index** (`sorted_idx_priority.npy`) from
   `sort_priority.pkl` so queries can avoid per-query sorts.
6. Document the interpretation of `listed_search` and the
   `filled in with EM` flags.

These artifacts are then consumed by a separate `EmmaDataset` / `EmmaSearcher`
wrapper (e.g. in `emma_core.py` and `emma_search.py`) which **only loads** them
and never recomputes them.

> **Important:** This notebook is meant to be run once to create the `.pkl` and
> `.npy` files in the same directory as the EMMa data. After that, you can
> treat those files as part of the dataset and just load them in your code.


## 1. Setup and Core Files

We assume the following files are present in the working directory (`ROOT`):

- `EMMa_v1.1.fdd` – main dataset
- `search_file.fdd` – search strings (title, description, features, materials, categories)
- `splits_v1.1.fdd` – train/val/test/held-out/calibration split ASINs
- `category_search_file.pkl`
- `weight_search_file.pkl`
- `price_search_file.pkl`
- `listed_search_file.pkl`
- `material_search_file.pkl`
- `image_search_file.pkl`
- `sort_priority.pkl`
- `custom_search_file.pkl`

We’ll load them here and establish a **canonical ASIN order** given by
`search.keys()`. All vectorized search arrays are assumed to follow this order.

⚠️ **Important:** `list(dataset.keys())` is **not** in the same
order as `list(search.keys())`. For all search-related arrays, we take
`asins = list(search.keys())` as the canonical order and build an index
mapping `asin2idx` from ASIN → row index.


In [ ]:
import os
import pickle
import numpy as np
from freeze_dried_data.freeze_dried_data_old import FDD

ROOT = None

dataset = FDD(os.path.join(ROOT, "EMMa_v1.1.fdd"))
search  = FDD(os.path.join(ROOT, "search_file.fdd"))
splits  = FDD(os.path.join(ROOT, "splits_v1.1.fdd"))

def load_pkl(name: str):
    with open(os.path.join(ROOT, name), "rb") as f:
        return pickle.load(f)

category_search = load_pkl("category_search_file.pkl")
weight_search   = load_pkl("weight_search_file.pkl")
price_search    = load_pkl("price_search_file.pkl")
listed_search   = load_pkl("listed_search_file.pkl")
material_search = load_pkl("material_search_file.pkl")
image_search    = load_pkl("image_search_file.pkl")
sort_priority   = load_pkl("sort_priority.pkl")
custom_search   = load_pkl("custom_search_file.pkl")

# Canonical ASIN order: search.keys()
asins = list(search.keys())
N = len(asins)
asin2idx = {a: i for i, a in enumerate(asins)}

print(f"Loaded {N:,} ASINs.")


## 2. Exploring File Contents

Before reverse-engineering anything, it's useful to look at a few concrete
examples and understand the structure of each file.

We’ll pick some ASINs we’ve looked at by hand (from previous debugging) that will help reverse engineering all of the required fields:

- `B01EL5834O` – dog jacket
- `B004LUQRJI` – hair styling paste
- `B001ECJPK4` – cactus outlet cover
- `B00EMP6EWM` – kitchen utensil set
- `B000KK6WZW` – Corvette speedometer cable


In [ ]:
example_asins = [
    "B01EL5834O",  # dog jacket
    "B004LUQRJI",  # hair styling paste
    "B001ECJPK4",  # cactus outlet cover
    "B00EMP6EWM",  # kitchen utensil / bluetooth speaker example from notes
    "B000KK6WZW",  # corvette speedometer cable
]

for asin in example_asins:
    print("ASIN:", asin)
    rec = dataset[asin]
    print("  category:", rec["category"])  # list of ints
    print("  price:", rec["price"])
    print("  mass:", rec.get("mass"))
    print("  ratings:", rec["ratings"])
    print("  custom:", rec["custom"])  # 5-dim vector
    print("  filled in with EM:", rec["filled in with EM"])  # [mask, b1, b2, b3, b4]
    print("  title:", rec["title"])
    print("  #features:", len(rec["features"]))
    print("  materials shape:", rec["materials"].shape)
    print("  package dimensions:", rec["package_dimensions"])
    print("  search string:")
    print("   ", search[asin][:400], "...")
    idx = asin2idx[asin]
    print("  listed_search byte (binary):", f"{listed_search[idx,0]:08b}")
    print("  custom_search byte (binary):", f"{custom_search[idx,0]:08b}")
    print("  material_search row shape:", material_search[idx].shape)
    print("  sort_priority:", sort_priority[asin])
    print("\n" + "-"*80 + "\n")


### 2.1 Structure of `search_file.fdd`

From the example ASINs, we can retrieve the following information:

In [ ]:
for asin in example_asins:
    print("ASIN:", asin)
    print("  dataset:")
    rec = dataset[asin]
    print("    category:", rec["category"])  # list of ints
    print("    title:", rec["title"])
    print("    description:", rec["description"])
    print("    #features:", len(rec["features"]))
    print("  search string:")
    print("   ", search[asin])
    print("\n" + "-"*80 + "\n")

In reality, the search strings were derived from the contents of the dataset, being possible to decompose them into:

- 1 × title
- 1 × long description
- `k = len(dataset[asin]['features'])` feature bullet points
- `m` material tokens
- `d` category tokens

Where `d` is the number of valid (non-negative) category IDs for that ASIN.

We can codify this as helper functions:

In [ ]:
def split_search(asin: str):
    """Split the search string for an ASIN into components by '|||'."""
    return search[asin].split('|||')

def get_category_ids(asin: str):
    cats = dataset[asin]['category']
    return [c for c in cats if c >= 0]

def get_search_segments(asin: str):
    """Return (title, description, features[], materials[], categories[])."""
    parts = split_search(asin)
    rec = dataset[asin]
    k = len(rec['features'])
    d = len(get_category_ids(asin))
    title = parts[0]
    desc  = parts[1]
    feat_start = 2
    feat_end   = 2 + k
    features = parts[feat_start:feat_end]
    cat_start = len(parts) - d
    materials = parts[feat_end:cat_start]
    categories = parts[cat_start:]
    return title, desc, features, materials, categories

asin_demo = example_asins[0]
title, desc, feats, mats, cats = get_search_segments(asin_demo)
print("ASIN", asin_demo)
print("  title:", title)
print("  #features:", len(feats))
print("  materials:", mats)
print("  categories:", cats)


We will reuse these helpers in the reverse-engineering sections below.


### 2.2 Structure of `listed_search.pkl`

From the example ASINs, we can retrieve the following information:

In [ ]:
for asin in example_asins:
    print("ASIN:", asin)
    rec = dataset[asin]
    print("  category:", rec["category"])  # list of ints
    print("  price:", rec["price"])
    print("  mass:", rec.get("mass"))
    print("  ratings:", rec["ratings"])
    idx = asin2idx[asin]
    print("  listed_search byte (binary):", f"{listed_search[idx,0]:08b}")
    print("\n" + "-"*80 + "\n")

From these outputs, it is possible to conclude that:

- **bit[0]** of 'listed_search' indicates whether the **categories** were filled by EM
- **bit[1]** of 'listed_search' indicates whether the **price** was filled by EM
- **bit[2]** of 'listed_search' indicates whether the **weight** was filled by EM
- 'listed_search' **does not** encode whether the **ratings** were filled or not by EM

## 3. Category Mapping: `(category_id, depth) → name`

Each dataset entry stores a list of category IDs:

```python
dataset[asin]['category']  # e.g. [12, 131, 1039, 3011, 3495, -1]
```

The negative IDs (typically `-1`) are padding. The non-negative IDs form a path
in the Amazon category tree, e.g. `[12, 131, 1039, 3011, 3495]`.

From §2.1 we know that the **last `d` tokens** of the search string are the
category names, where `d = number of valid category IDs`. Therefore, for each ASIN:

1. Extract non-negative category IDs.
2. Extract the last `d` tokens as category names.
3. For each depth `depth = 0, 1, ..., d-1`, pair `(cid, depth)` with the
   corresponding name.

Across the entire dataset we expect this mapping to be consistent; if we see a
collision (same `(cid, depth)` but different names), we log it.

Finally, we save the mapping as `cat_id2name.pkl` for use in the dataset
wrapper (which **only loads** this file and does not recompute it).


In [ ]:
from typing import Dict, Tuple, List

def get_category_names_from_search(asin: str) -> List[str]:
    parts = split_search(asin)
    ids = get_category_ids(asin)
    if not ids:
        return []
    k = len(ids)
    return parts[-k:]

cat_id2name: Dict[Tuple[int, int], str] = {}

total = len(asins)
for i, asin in enumerate(asins):
    if i % 10000 == 0:
        print(f"[cat] {i:,}/{total:,} ({100*i/total:.1f}%)")

    ids = get_category_ids(asin)
    if not ids:
        continue

    names = get_category_names_from_search(asin)
    if len(ids) != len(names):
        # inconsistent entry; skip
        continue

    for depth, (cid, cname) in enumerate(zip(ids, names)):
        key = (cid, depth)
        prev = cat_id2name.get(key)
        if prev is None:
            cat_id2name[key] = cname
        elif prev != cname:
            # collision: keep the first one and report
            print("category name collision:", key, repr(prev), "vs", repr(cname))

print(f"[cat] done, {len(cat_id2name)} unique (cid, depth) entries.")

# Save mapping
with open(os.path.join(ROOT, "cat_id2name.pkl"), "wb") as f:
    pickle.dump(cat_id2name, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Wrote cat_id2name.pkl")


## 4. Materials: Deriving the 182-Dim Mapping

Now we turn to the materials. For each ASIN we have:

- A **182-dimensional** probability vector in the main dataset:

  ```python
  dataset[asin]['materials']  # shape (182,)
  ```

- A **23-byte** row in `material_search`:

  ```python
  material_search[i]         # shape (23,) uint8
  ```

  If we unpack its bits, we get 23×8 = 184 bits, of which the first 182
  correspond to individual materials (the last 2 bits appear unused).

- A list of **material tokens** in `search_file.fdd`, obtained in §2.1 as the
  segment between features and categories.

Intuitively:

- `dataset[asin]['materials'][b]` is the model’s estimated probability that
  material `b` is present.
- `material_search[i]` encodes a **selected subset** of those materials as a
  bitfield (e.g. top-k or above threshold), where bit `b` is 1 if the
  corresponding material is selected for ASIN `i`.
- The **textual name** for material `b` should be one of the tokens in the
  materials segment of `search[asin]`.

We can discover the mapping by:

1. For each ASIN, extract the **set** of material tokens from the search
   string (lowercased, stripped).
2. For each bit index `b` in `0..181`, compute the Jaccard similarity between:
   - The set of ASINs where bit `b` is 1.
   - The set of ASINs where token `t` appears in the materials segment.
3. For each bit `b`, pick the token `t` with the highest Jaccard similarity.

Empirically, **every bit has a unique best token with Jaccard = 1.0**, which
means the mapping is exact.

Below we implement this derivation. This is the expensive part of the notebook
(it makes a single pass over all ASINs, but the nested counters can still take
a while), so you can run it once and then cache the resulting list.


In [ ]:
from collections import Counter, defaultdict

def material_bits_for_row(i: int) -> np.ndarray:
    """Return a (182,) 0/1 array of material bits for row i (asin index)."""
    row_bytes = material_search[i].astype(np.uint8)      # shape (23,)
    bits = np.unpackbits(row_bytes)                      # shape (184,)
    return bits[:182]

def get_material_tokens(asin: str):
    """Return the set of material tokens from the search string for this ASIN."""
    title, desc, feats, mats, cats = get_search_segments(asin)
    out = []
    for m in mats:
        t = m.strip().lower()
        if t:
            out.append(t)
    return set(out)

bit_on_count = np.zeros(182, dtype=np.int64)         # how many ASINs have bit b = 1
bit_token_joint = [Counter() for _ in range(182)]    # for each bit, counts over tokens
token_count = Counter()                              # how many ASINs contain token t

total = len(asins)
for i, asin in enumerate(asins):
    if i % 10000 == 0:
        print(f"[materials pass] {i:,}/{total:,} ({100*i/total:.1f}%)")

    bits = material_bits_for_row(i)  # (182,) 0/1
    token_set = get_material_tokens(asin)

    # Count tokens per ASIN once (set)
    for t in token_set:
        token_count[t] += 1

    on_bits = np.nonzero(bits)[0]
    for b in on_bits:
        bit_on_count[b] += 1
        for t in token_set:
            bit_token_joint[b][t] += 1

print(f"[materials pass] done: {total:,}/{total:,} (100.0%)")


Now, for each bit `b`, we compute the Jaccard similarity between the set of
ASINs where bit `b` is 1 and the set of ASINs where token `t` appears, using
the **aggregated counts** we just built:

In [ ]:
bit_to_best_token = {}
token_to_best_bit = {}

for b in range(182):
    if bit_on_count[b] == 0:
        continue
    best_tok = None
    best_score = 0.0
    for t, ct in bit_token_joint[b].items():
        # Jaccard(bit b, token t)
        union = bit_on_count[b] + token_count[t] - ct
        score = ct / union if union else 0.0
        if score > best_score:
            best_score = score
            best_tok = t
    bit_to_best_token[b] = (best_tok, best_score)

print("Top few bits and their best tokens:")
for b in range(10):
    print(b, bit_to_best_token.get(b))


In practice, inspecting the printed output shows entries of the form:

```text
0 ('18/10 steel', 1.0)
1 ('18/8 steel', 1.0)
2 ('abs', 1.0)
3 ('acrylic', 1.0)
...
181 ('zinc alloy', 1.0)
```

That is, **each bit has a unique token with Jaccard = 1.0**. The final
material name list can thus be constructed by taking the `best_tok` in order of increasing bit index.


In [ ]:
MATERIAL_NAMES = []
for b in range(182):
    tok, score = bit_to_best_token.get(b, (None, 0.0))
    if tok is None:
        raise ValueError(f"No token found for bit {b}")
    MATERIAL_NAMES.append(tok)

print("#materials:", len(MATERIAL_NAMES))
print("First 10:", MATERIAL_NAMES[:10])

with open(os.path.join(ROOT, "material_names.pkl"), "wb") as f:
    pickle.dump(MATERIAL_NAMES, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Wrote material_names.pkl")


### 4.1 Sanity check: probability distributions vs bits

We can also do a quick sanity check: for a material `b`, compare the average
probability `dataset[asin]['materials'][b]` when bit `b` is 1 vs when it is 0.
We expect the **mean probability to be much higher when the bit is 1**.


In [ ]:
for b in range(5):  # just a few bits
    vals_on = []
    vals_off = []
    for i, asin in enumerate(asins[:100000]):  # subsample for speed
        bits = material_bits_for_row(i)
        v = dataset[asin]['materials'][b]
        if bits[b]:
            vals_on.append(v)
        else:
            vals_off.append(v)
    print(
        "bit", b,
        "mean_on =", np.mean(vals_on) if vals_on else None,
        "mean_off =", np.mean(vals_off) if vals_off else None,
        "n_on", len(vals_on),
        "n_off", len(vals_off),
    )


In practice, you should see something like:

```text
bit 0 mean_on ≈ 0.78, mean_off ≈ 0.0047
bit 1 mean_on ≈ 0.81, mean_off ≈ 0.0029
...
```

which supports the interpretation that the bitfield is selecting materials
with high probability according to the distribution.


## 5. Custom Properties: Reverse-Engineering the Mapping

Each dataset entry has a 5-dimensional `custom` vector:

```python
dataset[asin]['custom']  # shape (5,)
```

The original paper suggests these correspond to:

- `sharp`
- `transparent`
- `deformable`
- `articulating`
- `electronic`

Additionally, the `custom_search` file contains a **single byte** per ASIN:

```python
custom_search[i, 0]  # uint8
```

If we unpack this byte into bits, we get 8 bits; correlating each bit with
each of the 5 custom probabilities across the full dataset reveals an almost
perfect one-to-one mapping:

- Bit 0 → sharp
- Bit 1 → transparent
- Bit 2 → deformable
- Bit 3 → articulating
- Bit 4 → electronic

Bits 5–7 appear unused (the corresponding probabilities are always zero, so
the correlation is undefined or zero).

Here we reproduce the correlation calculation and then fix the mapping.


In [ ]:
def custom_bits_for_idx(i: int) -> np.ndarray:
    value = int(custom_search[i, 0])
    bits = np.unpackbits(np.uint8(value))  # length 8
    return bits

N = len(asins)
custom_probs = np.zeros((N, 5), dtype=np.float32)
custom_bits  = np.zeros((N, 8), dtype=np.uint8)

for i, asin in enumerate(asins):
    if i % 10000 == 0:
        print(f"[custom] {i:,}/{N:,} ({100*i/N:.1f}%)")
    custom_probs[i] = dataset[asin]['custom']
    custom_bits[i]  = custom_bits_for_idx(i)

print(f"[custom] done: {N:,}/{N:,} (100.0%)")

names = ["sharp", "transparent", "deformable", "articulating", "electronic"]
for bit in range(8):
    print(f"Bit {bit}")
    x = custom_bits[:, bit]
    for j, name in enumerate(names):
        y = custom_probs[:, j]
        # Pearson correlation; may produce warnings if y is constant
        corr = np.corrcoef(x, y)[0, 1]
        print("   ", name, corr)
    print()


Typical output (abbreviated):

```text
Bit 0
    sharp        ≈ 0.91
    transparent  ≈ 0.01
    deformable   ≈ -0.10
    articulating ≈ 0.05
    electronic   ≈ -0.01

Bit 1
    sharp        ≈ -0.02
    transparent  ≈ 0.88
    ...

Bit 2
    deformable   ≈ 0.92
    ...

Bit 3
    articulating ≈ 0.91
    ...

Bit 4
    electronic   ≈ 0.98
```

This confirms the mapping. We then fix the custom names in that order and
write them to `custom_names.pkl`.


In [ ]:
CUSTOM_NAMES = ["sharp", "transparent", "deformable", "articulating", "electronic"]

with open(os.path.join(ROOT, "custom_names.pkl"), "wb") as f:
    pickle.dump(CUSTOM_NAMES, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Wrote custom_names.pkl")


## 6. Global Sort Index from `sort_priority`

The file `sort_priority.pkl` is a dictionary:

```python
sort_priority[asin] -> float score
```

The keys match the full set of ASINs and the values are some global ranking
score (e.g. item popularity / quality). We don’t need to interpret the score;
we just need a **stable global ordering** so the search backend can avoid
sorting from scratch in every query.

To do this, we:

1. Build a vector `scores[i] = sort_priority[asins[i]]` in canonical ASIN order.
2. Compute `idx_sorted = np.argsort(scores)[::-1]` (descending).
3. Save `idx_sorted` to `sorted_idx_priority.npy`.

At query time, we can take a boolean mask of candidate items and filter this
pre-sorted index array instead of sorting per-query.


In [ ]:
scores = np.empty(N, dtype=np.float32)
for i, a in enumerate(asins):
    scores[i] = sort_priority[a]

idx_sorted = np.argsort(scores)[::-1]
np.save(os.path.join(ROOT, "sorted_idx_priority.npy"), idx_sorted)

print("Wrote sorted_idx_priority.npy")


## 7. Listed vs EM-Predicted Data

The `listed_search` file contains a single byte per ASIN:

```python
listed_search[i, 0]  # uint8
```

Unpacking this byte into bits yields 8 bits. The main dataset also includes a
`filled in with EM` field for each ASIN:

```python
filled = dataset[asin]['filled in with EM']
materials_mask = filled[0]  # 23-byte mask (uint8) indicating which materials
                            # were predicted by the EM model
category_em   = filled[1]   # bool: was category filled by EM?
price_em      = filled[2]   # bool: was price filled by EM?
mass_em       = filled[3]   # bool: was mass filled by EM?
ratings_em    = filled[4]   # bool: were ratings filled by EM?
```

By correlating the bits in `listed_search` with the EM booleans above across
many ASINs, we find that:

- **Bit 0** corresponds perfectly (with sign flip) to `category_em`.
- **Bit 1** corresponds perfectly to `price_em`.
- **Bit 2** corresponds perfectly to `mass_em`.
- No bit appears to correspond to `ratings_em`.

Interpretation:

- If `bit 0 == 1`, the *category* was **listed** (not predicted by EM).
- If `bit 1 == 1`, the *price* was listed.
- If `bit 2 == 1`, the *mass* was listed.

Conversely, `category_em == True` means the category was predicted by EM, and
`bit 0` will be 0 (and similarly for price/mass).

We don’t need to create any extra files for this; the dataset wrapper can
decode these bits and flags directly when applying filters like “must not be
predicted by EM (must be listed)”. Below is the correlation code used to
verify this mapping.


In [ ]:
def listed_bits_for_idx(i: int) -> np.ndarray:
    value = int(listed_search[i, 0])
    return np.unpackbits(np.uint8(value))  # 8 bits

listed_bits = np.zeros((N, 8), dtype=np.uint8)
em_flags   = np.zeros((N, 4), dtype=np.uint8)  # category, price, mass, ratings

for i, asin in enumerate(asins):
    if i % 10000 == 0:
        print(f"[listed] {i:,}/{N:,} ({100*i/N:.1f}%)")
    listed_bits[i] = listed_bits_for_idx(i)
    _, cat_em, price_em, mass_em, ratings_em = dataset[asin]['filled in with EM']
    em_flags[i] = [cat_em, price_em, mass_em, ratings_em]

print(f"[listed] done: {N:,}/{N:,} (100.0%)")

for bit in range(8):
    print("Bit", bit)
    x = listed_bits[:, bit]
    for j, attr in enumerate(["category_em", "price_em", "mass_em", "ratings_em"]):
        y = em_flags[:, j]
        # Pearson correlation; may be NaN if y is constant (e.g. always 0)
        corr = np.corrcoef(x, y)[0, 1]
        print("  vs", attr, "=", corr)
    print()


Typical output (schematically):

```text
Bit 0
  vs category_em ≈ -1.0
  vs price_em    ≈ small
  vs mass_em     ≈ 0.23
  vs ratings_em  ≈ -0.87

Bit 1
  vs category_em ≈ -0.06
  vs price_em    ≈ -1.0
  vs mass_em     ≈ -0.11
  vs ratings_em  ≈ small

Bit 2
  vs category_em ≈ 0.23
  vs price_em    ≈ -0.11
  vs mass_em     ≈ -1.0
  vs ratings_em  ≈ 0.30
```

The strong (±1) correlations along the diagonal show that bits 0–2 directly
encode whether the corresponding attribute was listed vs EM-predicted.

Again, we **do not** create any file for this; we just record the
interpretation so the dataset/search wrapper can expose filters like:

- `must_not_be_predicted_by_em_category`
- `must_not_be_predicted_by_em_price`
- `must_not_be_predicted_by_em_mass`

by combining `listed_search` and/or the `filled in with EM` booleans.


## 8. Summary of Generated Artifacts

Running this notebook end-to-end generates the following files (in `ROOT`):

- `cat_id2name.pkl`
  - Python dict: `(category_id: int, depth: int) -> category_name: str`
- `material_names.pkl`
  - Python list of length 182: `material_names[bit_index] = token_str`
- `custom_names.pkl`
  - Python list of length 5: `["sharp", "transparent", "deformable", "articulating", "electronic"]`
- `sorted_idx_priority.npy`
  - NumPy array of shape `(N,)` with ASIN indices sorted by descending `sort_priority`

These are treated as **inputs** by the separate dataset/search modules
(e.g. `emma_core.py`, `emma_search.py`). Those modules **only load** these
artifacts; they do not recompute them.

This keeps the runtime search code clean and fast, while documenting the
reverse-engineering process here in the notebook.
